In [ ]:
import sys
from pathlib import Path
ROOT = Path("/home/jovyan/work/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import get_spark
from src.io import read_parquet, write_parquet, to_csv_via_pandas

from src.data_split import align_and_join_datasets, get_client_id_sets, get_solo_datasets
import src.config as cfg

In [ ]:
spark = get_spark("UnionDatasets")


df_beh_clean = read_parquet(spark, cfg.BEHAVIOURAL_PATH_CLEAN)
df_cli_clean = read_parquet(spark, cfg.CLIENTS_PATH_CLEAN)

In [3]:
id_sets = get_client_id_sets(df_beh_clean, df_cli_clean)

print(f"IDs Comunes: {id_sets['comunes'].count()}")
print(f"IDs Solo en Clientes: {id_sets['solo_cli'].count()}")
print(f"IDs Solo en Behavioral: {id_sets['solo_beh'].count()}")

IDs Clientes (únicos): 154207
IDs Behavioral (únicos): 46046
IDs Comunes: 45668
IDs Comunes: 45668
IDs Solo en Clientes: 108539
IDs Solo en Behavioral: 378


In [4]:
df_combined = align_and_join_datasets(df_beh_clean, df_cli_clean)

df_combined.printSchema()
print(f"Filas en el dataset combinado (INNER JOIN): {df_combined.count()}")

root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: integer (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: double (nullable = true)
 |-- AMOUNT_PRODUCT: double (nullable = true)
 |-- INSTALLMENT: double (nullable = true)
 |-- EDUCATION: string (nullable = true)
 |-- MARITAL_STATUS: string (nullable = true)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: double (nullable = true)
 |-- AGE_IN_YEARS: double (nullable = true)
 |-- JOB_SENIORITY: double (nullable = true)
 |-- HOME_SENIORITY: double (nullable = true)
 |-- LAST_UPDATE: double (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- FAMILY_SIZE: double (nullable = true)
 |-- PROACTIVE_SCORING: double (nullable = true)
 |-- BEHAVIORAL_SCORING: double (nullable = true)
 |-- DAYS_LAST_INFO_CHANGE: double (nullable = true)
 |-- NUMBER_OF_PRODUCTS: double (nullable = true)
 |-- OCCUPATION: string (nulla

In [5]:
write_parquet(df_combined, cfg.COMBINED_PATH, mode="overwrite")

to_csv_via_pandas(df_combined, cfg.COMBINED_CSV, index=False)

In [6]:
solos = get_solo_datasets(df_beh_clean, df_cli_clean)

write_parquet(solos[1], cfg.SOLO_BEH_PATH, mode="overwrite")
write_parquet(solos[0], cfg.SOLO_CLI_PATH, mode="overwrite")

to_csv_via_pandas(solos[1], cfg.SOLO_BEH_CSV, index=False)
to_csv_via_pandas(solos[0], cfg.SOLO_CLI_CSV, index=False)

IDs Clientes (únicos): 154207
IDs Behavioral (únicos): 46046
IDs Comunes: 45668
